In [1]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF

In [2]:
df = pd.read_csv('datafiles/train.csv')

In [3]:
#欠損値の確認
for c in df.columns:
    null_counts = df[c].isnull().sum()
    if null_counts != 0:
        print(f'{null_counts}  列＝{c}')

259  列＝LotFrontage
1369  列＝Alley
872  列＝MasVnrType
8  列＝MasVnrArea
37  列＝BsmtQual
37  列＝BsmtCond
38  列＝BsmtExposure
37  列＝BsmtFinType1
38  列＝BsmtFinType2
1  列＝Electrical
690  列＝FireplaceQu
81  列＝GarageType
81  列＝GarageYrBlt
81  列＝GarageFinish
81  列＝GarageQual
81  列＝GarageCond
1453  列＝PoolQC
1179  列＝Fence
1406  列＝MiscFeature


In [4]:
#明らかに不要な'id'を除く
df = df.drop(['Id'], axis = 1)

In [5]:
#ダミー変数化する行の抜き出し
to_dummy_cols = []
for c in df.columns:
    if type(c) == str:
        to_dummy_cols.append(c)
print(to_dummy_cols)

['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fen

In [6]:
'''
#strの特徴量の中にNAが混ざっている列
Alley
MasVnrType
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2
Electrical
FireplaceQu
GarageType
GarageFinish
GarageQual
GarageCond
PoolQC
Fence
MiscFeature

#intの特徴量の中にNAが混ざっている列
LotFrontage  NAを0に変更
MasVnrArea   NAを0に変更
GarageYrBlt  NAを0に変更
'''
#data_description.txt を確認すると、すべての特徴量で'NA'に意味があるようだったので補完
to_NA_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 
    'GarageCond', 'Fence', 'MiscFeature'
]
df[to_NA_cols] = df[to_NA_cols].fillna('NA')
df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0.0)


In [7]:
print(df.columns)

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [8]:
#float型に変更
not_to_dummy = ['MSSubClass', 'LotFrontage', 
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
    'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]

In [9]:
df[not_to_dummy] = df[not_to_dummy].astype('float64')

In [10]:
#ダミー変数化
to_dummy = set(df.columns) - set(not_to_dummy)
to_dummy = list(to_dummy)
for c in to_dummy:
    dummy = pd.get_dummies(df[c], prefix=c, drop_first = True, dtype = int)
    df = pd.concat([df, dummy], axis = 1)
    df = df.drop([c], axis = 1)

In [11]:
#float型に変更
df = df.astype('float64')

In [12]:
#説明変数と目的変数のデータフレームを作る
df_y = pd.DataFrame(df['SalePrice'])
df_x = df.drop(['SalePrice'], axis = 1)

#標準化
sc_model=StandardScaler()
sc_model.fit(df_x)
sc_x = sc_model.fit_transform(df_x)

In [13]:
#重回帰、リッジ回帰、ラッソ回帰、回帰木を実践し、結果を比較する。
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [14]:
#重回帰
model1 = LinearRegression()
result = cross_validate(model1, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel1のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel1のスコア＝0.5498616628567794


In [15]:
#リッジ回帰
#正則化項の定数を0.01~20まで検証
best_ridgescore = 0
best_alpha = 0

#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    ridgeModel = Ridge(random_state = 0, alpha = i)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = i
print(f'正則化項＝{best_alpha}　リッジ回帰のスコア＝{best_ridgescore}')

#完成したリッジ回帰モデルで学習
model2 = Ridge(alpha = best_alpha)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

正則化項＝100　リッジ回帰のスコア＝0.77737728318019
完成したmodel2のスコア＝0.77737728318019


In [16]:
#ラッソ回帰
#正則化項の定数を0.01~20まで検証
best_lassoscore = 0
best_alpha = 0
#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    lassoModel = Lasso(random_state = 0, alpha = i)
    all_result = cross_validate(lassoModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = i
print(f'正則化項＝{best_alpha}　ラッソ回帰のスコア＝{best_lassoscore}')

#完成したラッソ回帰モデルで学習
model3 = Lasso(alpha = best_alpha)
result = cross_validate(model3, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel3のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.501e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.960e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.407e+11, toleranc

正則化項＝100　ラッソ回帰のスコア＝0.6355601632269483
完成したmodel3のスコア＝0.6355601632269483


In [17]:
'''
単純なモデルでは、下記のような結果となった。

完成したmodel1のスコア＝0.5498616628567818
完成したmodel2のスコア＝0.7773772831801898
完成したmodel3のスコア＝0.6355611660711009

以下では、特徴量を改善したモデルでも実験する。
'''

'\n単純なモデルでは、下記のような結果となった。\n\n完成したmodel1のスコア＝0.5498616628567818\n完成したmodel2のスコア＝0.7773772831801898\n完成したmodel3のスコア＝0.6355611660711009\n\n以下では、特徴量を改善したモデルでも実験する。\n'

In [18]:
model2.fit(sc_x, df_y)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [19]:
#リッジ回帰の係数と切片の確認
coef_df = pd.DataFrame({
    'col': df_x.columns,
    'coef': model2.coef_
})
print(f'係数: {coef_df }')
print(f'切片: {model2.intercept_}')

係数:                   col          coef
0          MSSubClass  -3991.979995
1         LotFrontage  -1008.982281
2             LotArea   5162.279218
3         OverallQual  11149.951899
4         OverallCond   5368.292195
..                ...           ...
254  MiscFeature_Shed     93.718434
255  MiscFeature_TenC   -626.341810
256  Utilities_NoSeWa  -1266.498393
257         PoolQC_Fa  -4014.386787
258         PoolQC_Gd  -9399.127407

[259 rows x 2 columns]
切片: [180921.19589041]


In [20]:
coef_df.sort_values('coef', ascending=False)

,col,coef
15,GrLivArea,11219.307651
3,OverallQual,11149.951899
32,PoolArea,8982.824397
240,RoofMatl_WdShngl,8745.629193
13,2ndFlrSF,8219.109523
...,...,...
208,Condition2_PosN,-6044.031216
99,KitchenQual_TA,-7375.819498
143,BsmtQual_Gd,-7505.201374
98,KitchenQual_Gd,-7927.974914


In [21]:
'''
NAが複数行にかけて同一の意味をもつもの
・no basementの意味
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2

・no garageの意味
GarageType
GarageFinish
GarageQual
GarageCond
'''
sc_x = pd.DataFrame(sc_x)
sc_x.columns = df_x.columns
for c in sc_x.columns:
    print(c)


MSSubClass
LotFrontage
LotArea
OverallQual
OverallCond
YearBuilt
YearRemodAdd
MasVnrArea
BsmtFinSF1
BsmtFinSF2
BsmtUnfSF
TotalBsmtSF
1stFlrSF
2ndFlrSF
LowQualFinSF
GrLivArea
BsmtFullBath
BsmtHalfBath
FullBath
HalfBath
BedroomAbvGr
KitchenAbvGr
TotRmsAbvGrd
Fireplaces
GarageYrBlt
GarageCars
GarageArea
WoodDeckSF
OpenPorchSF
EnclosedPorch
3SsnPorch
ScreenPorch
PoolArea
MiscVal
MoSold
YrSold
Neighborhood_Blueste
Neighborhood_BrDale
Neighborhood_BrkSide
Neighborhood_ClearCr
Neighborhood_CollgCr
Neighborhood_Crawfor
Neighborhood_Edwards
Neighborhood_Gilbert
Neighborhood_IDOTRR
Neighborhood_MeadowV
Neighborhood_Mitchel
Neighborhood_NAmes
Neighborhood_NPkVill
Neighborhood_NWAmes
Neighborhood_NoRidge
Neighborhood_NridgHt
Neighborhood_OldTown
Neighborhood_SWISU
Neighborhood_Sawyer
Neighborhood_SawyerW
Neighborhood_Somerst
Neighborhood_StoneBr
Neighborhood_Timber
Neighborhood_Veenker
PavedDrive_P
PavedDrive_Y
BsmtCond_Gd
BsmtCond_NA
BsmtCond_Po
BsmtCond_TA
Condition1_Feedr
Condition1_Norm
Condit

In [22]:
'''
前のセルの結果、下記のセルが重複するNAのダミー変数化であった。

BsmtQual_NA
BsmtCond_NA
BsmtExposure_NA
BsmtFinType1_NA
BsmtFinType2_NA
　→BsmtQual_NAのみ残し、他は削除

GarageCond_NA
GarageType_NA
GarageQual_NA
　→GarageFinish列で同一の意味を表せるので削除
'''
to_drop = ['BsmtCond_NA', 'BsmtExposure_NA', 'BsmtFinType1_NA', 'BsmtFinType2_NA', 'GarageCond_NA', 'GarageType_NA', 'GarageQual_NA']
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

In [23]:
#改善したデータフレームを用いて、リッジ回帰モデルで学習
result = cross_validate(model1, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel1のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

result = cross_validate(model3, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel3のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel1のスコア＝0.5527321274460041
完成したmodel2のスコア＝0.7774453479883098


ValueError: not enough values to unpack (expected at least 1, got 0)

ValueError: not enough values to unpack (expected at least 1, got 0)

ValueError: not enough values to unpack (expected at least 1, got 0)

完成したmodel3のスコア＝0.6353598643525149


In [24]:
#回帰木
best_score = 0
best_depth = 0
#深さを１～30まで実験
for i in range(1,31):
    model = DecisionTreeRegressor(max_depth = i, random_state = 0)
    all_result = cross_validate(model, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_score:
        best_score = result
        best_depth = i
print(f'深さ＝{best_depth}　回帰木のスコア＝{best_score}')

model4 = DecisionTreeRegressor(max_depth = best_depth, random_state = 0)
result = cross_validate(model4, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel4のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

深さ＝6　回帰木のスコア＝0.7939800342838383
完成したmodel4のスコア＝0.7939800342838383


In [25]:
'''
特徴量修正後は下記のような結果となった。単純な特徴量のモデルによる前回の結果からの変化はわずかであった。


完成したmodel1のスコア＝0.5527321274460049
完成したmodel2のスコア＝0.7774453479883086
完成したmodel3のスコア＝0.6353585442252603
完成したmodel4のスコア＝0.7904883919255022

以下では、さらに特徴量を改善したモデルでも実験する。
'''

'\n特徴量修正後は下記のような結果となった。単純な特徴量のモデルによる前回の結果からの変化はわずかであった。\n\n\n完成したmodel1のスコア＝0.5527321274460049\n完成したmodel2のスコア＝0.7774453479883086\n完成したmodel3のスコア＝0.6353585442252603\n完成したmodel4のスコア＝0.7904883919255022\n\n以下では、さらに特徴量を改善したモデルでも実験する。\n'

In [26]:
#多重共線性の解消
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
vif_df.sort_values('VIF_Factor', ascending=False)

c:\Users\natsu\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,VIF_Factor,features
9,inf,BsmtFinSF2
10,inf,BsmtUnfSF
157,inf,Exterior2nd_CBlock
15,inf,GrLivArea
14,inf,LowQualFinSF
...,...,...
225,1.228599,SaleType_Oth
30,1.227674,3SsnPorch
220,1.205392,SaleType_Con
71,1.193595,Condition1_RRNe


In [27]:
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

,VIF_Factor,features
8,inf,BsmtFinSF1
9,inf,BsmtFinSF2
10,inf,BsmtUnfSF
11,inf,TotalBsmtSF
12,inf,1stFlrSF
13,inf,2ndFlrSF
14,inf,LowQualFinSF
15,inf,GrLivArea
24,2178.616803,GarageYrBlt
81,inf,Exterior1st_CBlock


In [28]:
sc_x[df_high_vif['features']].corr()

,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,GarageYrBlt,Exterior1st_CBlock,Exterior1st_VinylSd,RoofStyle_Gable,RoofStyle_Hip,GarageQual_TA,Exterior2nd_CBlock,Exterior2nd_VinylSd,GarageCond_TA,GarageFinish_NA,MiscFeature_NA,MiscFeature_Shed
BsmtFinSF1,1.000000,-0.050117,-0.495251,0.522396,0.445863,-0.137079,-0.064503,0.208171,0.115843,-0.007386,-0.025470,-0.193130,0.213285,0.166765,-0.007386,-0.025469,0.153382,-0.108043,0.009811,-0.009804
BsmtFinSF2,-0.050117,1.000000,-0.209294,0.104810,0.097117,-0.099260,0.014807,-0.009640,0.035070,0.009489,-0.116327,-0.070168,0.033034,0.059823,0.009489,-0.124406,0.035065,-0.039733,0.015026,-0.026293
BsmtUnfSF,-0.495251,-0.209294,1.000000,0.415360,0.317987,0.004469,0.028167,0.240257,0.042720,-0.008727,0.240121,-0.027912,0.044537,-0.015813,-0.008727,0.240813,0.022623,-0.032924,0.053208,-0.045992
TotalBsmtSF,0.522396,0.104810,0.415360,1.000000,0.819530,-0.174512,-0.033245,0.454868,0.176359,-0.012980,0.172596,-0.254702,0.278745,0.179447,-0.012980,0.170324,0.195143,-0.160098,0.069316,-0.066184
1stFlrSF,0.445863,0.097117,0.317987,0.819530,1.000000,-0.202646,-0.014241,0.566024,0.166642,-0.021856,0.070483,-0.314131,0.323994,0.171897,-0.021856,0.071504,0.186819,-0.154846,0.049235,-0.044764
2ndFlrSF,-0.137079,-0.099260,0.004469,-0.174512,-0.202646,1.000000,0.063353,0.687501,0.064402,0.011219,0.109556,0.086670,-0.113568,0.007144,0.011219,0.108558,0.034054,-0.060821,0.013603,-0.028513
LowQualFinSF,-0.064503,0.014807,0.028167,-0.033245,-0.014241,0.063353,1.000000,0.134683,-0.146467,-0.003148,-0.065438,0.024430,-0.018588,-0.124573,-0.003148,-0.063861,-0.130453,0.145128,-0.032285,0.017966
GrLivArea,0.208171,-0.009640,0.240257,0.454868,0.566024,0.687501,0.134683,1.000000,0.162543,-0.007050,0.136809,-0.156842,0.142294,0.120870,-0.007050,0.136877,0.153658,-0.151015,0.044534,-0.054956
GarageYrBlt,0.115843,0.035070,0.042720,0.176359,0.166642,0.064402,-0.146467,0.162543,1.000000,0.005672,0.100092,-0.054480,0.068337,0.729196,0.005672,0.096430,0.770768,-0.998601,0.002944,0.009322
Exterior1st_CBlock,-0.007386,0.009489,-0.008727,-0.012980,-0.021856,0.011219,-0.003148,-0.007050,0.005672,1.000000,-0.019327,0.013843,-0.012922,0.008826,1.000000,-0.019009,0.008322,-0.006345,0.005131,-0.004879


In [29]:
vif_df[vif_df["VIF_Factor"] == float("inf")]

,VIF_Factor,features
8,inf,BsmtFinSF1
9,inf,BsmtFinSF2
10,inf,BsmtUnfSF
11,inf,TotalBsmtSF
12,inf,1stFlrSF
13,inf,2ndFlrSF
14,inf,LowQualFinSF
15,inf,GrLivArea
81,inf,Exterior1st_CBlock
157,inf,Exterior2nd_CBlock


In [30]:
to_drop = ['BsmtFinSF2', '2ndFlrSF', 'Exterior2nd_CBlock']
sc_x[to_drop]

,BsmtFinSF2,2ndFlrSF,Exterior2nd_CBlock
0,-0.288653,1.161852,-0.02618
1,-0.288653,-0.795163,-0.02618
2,-0.288653,1.189351,-0.02618
3,-0.288653,0.937276,-0.02618
4,-0.288653,1.617877,-0.02618
...,...,...,...
1455,-0.288653,0.795198,-0.02618
1456,0.722112,-0.795163,-0.02618
1457,-0.288653,1.844744,-0.02618
1458,6.092188,-0.795163,-0.02618


In [31]:
#試しにいくつかの変数をdropしてみる
to_drop = ['BsmtFinSF2', '2ndFlrSF', 'Exterior2nd_CBlock']
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

In [32]:
#drop後のvifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

result = cross_validate(model4, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel4のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

#この結果により、vifが無限の変数はなくなった。しかし依然として多重共線性は高いため、今後も多重共線性の解消を目指す。

,VIF_Factor,features
22,2178.616803,GarageYrBlt
87,115.837730,Exterior1st_VinylSd
109,165.010225,RoofStyle_Gable
111,153.477931,RoofStyle_Hip
131,233.699047,GarageQual_TA
163,105.805972,Exterior2nd_VinylSd
186,283.746702,GarageCond_TA
213,2202.752816,GarageFinish_NA
242,960.199106,MiscFeature_NA
244,802.650747,MiscFeature_Shed


完成したmodel4のスコア＝0.7643184652780491


In [37]:
#データフレームを保存
df = pd.concat([sc_x, df_y], axis = 1)
df.to_csv('df.csv', index=False)

In [ ]:
#利便性上の理由から、以降の分析は他のファイルで行う。